# Geração da Base Simulada (People Analytics)
Este notebook gera a base simulada do projeto *Turnover Crítico* em uma instituição financeira fictícia com aproximadamente 2.500 colaboradores.


A base é criada com **intencionalidade analítica**, simulando padrões reais observados no mercado:
- Turnover elevado em Comercial e Tecnologia
- Queda de engajamento ao longo do tempo
- Desigualdade sutil em promoções para análise de equidade
- Custos de reposição altos em níveis estratégicos
- Relações realistas entre performance, engajamento e risco de saída

Todos os dados seguem princípios de:
- Reprodutibilidade (seed fixa)
- Ética e privacidade (dados 100% sintéticos)
- Estrutura de People Analytics corporativo

## 1. Imports e Configurações (Python)

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# Seed para reprodutibilidade
np.random.seed(42)

## 2. Geração da Tabela dim_employees
*(cadastro base dos 2.500 colaboradores)*

**Racional de negócio**
- Precisamos de atributos que expliquem promoções, engajamento, turnover.
- Estrutura similar à de um RH corporativo real.

**Racional técnico**
- Distribuições controladas por área, nível e salários.

In [2]:
# Número de colaboradores
n_employees = 2500

employee_id = np.arange(1, n_employees + 1)

# Distribuição por área
areas = ["Comercial", "Tecnologia", "Risco & Compliance", "Crédito", "Produtos Digitais", "Backoffice"]
area_prob = [0.28, 0.22, 0.12, 0.10, 0.13, 0.15]  # distribuição realista

employee_area = np.random.choice(areas, size=n_employees, p=area_prob)

# Níveis de cargo
niveis = ["Júnior", "Pleno", "Sênior", "Líder"]
nivel_prob = [0.35, 0.40, 0.20, 0.05]  # pirâmide organizacional

employee_nivel = np.random.choice(niveis, size=n_employees, p=nivel_prob)

# Gênero e raça para fairness
generos = ["F", "M", "Outro"]
genero_prob = [0.47, 0.50, 0.03]

racas = ["Branca", "Parda", "Preta", "Amarela", "Indígena", "Não declarado"]
raca_prob = [0.46, 0.38, 0.09, 0.04, 0.01, 0.02]

employee_genero = np.random.choice(generos, size=n_employees, p=genero_prob)
employee_raca = np.random.choice(racas, size=n_employees, p=raca_prob)

# Data de admissão simulada entre 2015 e 2025
admissao_start = datetime(2015, 1, 1)
employee_data_adm = [
    admissao_start + timedelta(days=int(np.random.uniform(0, 365 * 10)))
    for _ in range(n_employees)
]

# Idade
employee_idade = np.random.normal(loc=34, scale=6, size=n_employees).astype(int)

# Salário base inicial (modulado por área + nível)
base_salary_map = {
    "Júnior": 3000,
    "Pleno": 6000,
    "Sênior": 10000,
    "Líder": 15000
}

area_multiplier = {
    "Comercial": 1.0,
    "Tecnologia": 1.25,
    "Produtos Digitais": 1.35,
    "Risco & Compliance": 1.10,
    "Crédito": 1.15,
    "Backoffice": 0.95
}

employee_salario = [
    base_salary_map[nivel] * area_multiplier[area] * np.random.uniform(0.85, 1.15)
    for nivel, area in zip(employee_nivel, employee_area)
]

# Compilação
dim_employees = pd.DataFrame({
    "employee_id": employee_id,
    "area": employee_area,
    "nivel": employee_nivel,
    "genero": employee_genero,
    "raca": employee_raca,
    "data_admissao": employee_data_adm,
    "idade": employee_idade,
    "salario_base_inicial": employee_salario
})

dim_employees.head()

,employee_id,area,nivel,genero,raca,data_admissao,idade,salario_base_inicial
0,1,Tecnologia,Sênior,F,Parda,2018-09-25,37,11366.628604
1,2,Backoffice,Sênior,M,Parda,2018-04-30,27,10410.136524
2,3,Produtos Digitais,Pleno,M,Branca,2016-10-04,34,9098.936870
3,4,Risco & Compliance,Líder,F,Branca,2021-01-25,33,18049.780472
4,5,Comercial,Júnior,M,Parda,2019-10-06,35,2824.598271


## 3. Geração da fact_monthly_hr
*(employee-mês, com engajamento, performance, workload)*

**Racional de negócio**
- Engajamento cai antes do turnover
- Performance modula promoções
- Workload influencia burnout

**Racional técnico**
- Criar 24 meses * 2500 colaboradores = 60 mil linhas.

In [3]:
# Período de 24 meses
dates = pd.date_range(start="2024-01-01", end="2025-12-01", freq="MS")
n_months = len(dates)

records = []

for date in dates:
    for i in range(n_employees):
        
        area = dim_employees.loc[i, "area"]
        
        # Engajamento: queda intencional em Comercial e TI
        base_engajamento = np.random.normal(75, 10)
        
        if area in ["Comercial", "Tecnologia"]:
            meses_passados = (date.year - 2024) * 12 + date.month
            base_engajamento -= meses_passados * 0.7
        
        engajamento = max(0, min(100, base_engajamento))
        
        # Performance depende levemente da senioridade
        nivel = dim_employees.loc[i, "nivel"]
        perf_shift = {"Júnior": -0.2, "Pleno": 0, "Sênior": 0.2, "Líder": 0.3}
        
        performance = np.clip(np.random.normal(3 + perf_shift[nivel], 0.6), 1, 5)
        
        # Workload
        horas_extras = max(0, np.random.normal(12, 8))
        if area == "Tecnologia":
            horas_extras += np.random.uniform(4, 12)
        
        records.append([
            dim_employees.loc[i, "employee_id"],
            date,
            engajamento,
            performance,
            horas_extras
        ])

fact_monthly_hr = pd.DataFrame(records,
                               columns=["employee_id", "mes_ano", 
                                        "engajamento", "performance", 
                                        "horas_extras"])

## 4. Promoções (com viés sutil para teste de equidade)

**Intencionalidade:**
- Menor probabilidade de promoção para mulheres em Tecnologia (≈ −15%)

In [4]:
promo_records = []

for i in range(n_employees):
    genero = dim_employees.loc[i, "genero"]
    area = dim_employees.loc[i, "area"]
    nivel = dim_employees.loc[i, "nivel"]
    
    # Probabilidade base
    prob_prom = {"Júnior": 0.20, "Pleno": 0.15, "Sênior": 0.10, "Líder": 0.02}[nivel]
    
    # Viés controlado
    if area == "Tecnologia" and genero == "F":
        prob_prom *= 0.85
    
    # Simular se promoveu
    if np.random.rand() < prob_prom:
        new_nivel = {"Júnior": "Pleno", "Pleno": "Sênior", "Sênior": "Líder", "Líder": "Líder"}[nivel]
        
        promo_records.append([
            dim_employees.loc[i, "employee_id"],
            np.random.choice(dates),
            nivel,
            new_nivel
        ])

events_promotions = pd.DataFrame(promo_records,
                                 columns=["employee_id", "data",
                                          "nivel_anterior", "nivel_novo"])

## 5. Desligamentos (turnover problemático)

**Intencionalidade:**   
- Maior probabilidade em Comercial e TI
- Engajamento baixo aumenta risco
- Custos de reposição dependem do cargo

In [5]:
exit_records = []

for i in range(n_employees):
    area = dim_employees.loc[i, "area"]
    nivel = dim_employees.loc[i, "nivel"]
    
    # Probabilidades base
    base_prob = 0.08 if area in ["Comercial", "Tecnologia"] else 0.04
    
    # Risco adicional por engajamento baixo
    eng_media = fact_monthly_hr[fact_monthly_hr.employee_id == i+1]["engajamento"].mean()
    if eng_media < 60:
        base_prob *= 1.5
    
    # Simulação
    if np.random.rand() < base_prob:
        data_saida = np.random.choice(dates)
        tipo = np.random.choice(["Voluntário", "Involuntário"], p=[0.7, 0.3])
        
        custo = (
            base_salary_map[nivel] 
            * area_multiplier[area] 
            * np.random.uniform(2.5, 4.5)
        )
        
        exit_records.append([
            dim_employees.loc[i, "employee_id"],
            data_saida, tipo, custo
        ])

events_exits = pd.DataFrame(exit_records,
                            columns=["employee_id", "data_desligamento",
                                     "tipo", "custo_reposicao"])

## 6. Exportar arquivos

In [6]:
dim_employees.to_csv("../data/raw/dim_colaboradores.csv", index=False)
fact_monthly_hr.to_csv("../data/raw/fato_rh_mensal.csv", index=False)
events_promotions.to_csv("../data/raw/eventos_promocoes.csv", index=False)
events_exits.to_csv("../data/raw/eventos_desligamentos.csv", index=False)

print("Arquivos gerados e gravados com sucesso!")

Arquivos gerados e gravados com sucesso!
